
# LDAP Viewer (read-only, release-aware)

Purpose: **trust but verify** the LDAP raw CSVs before ETL. This notebook inspects headers, checks consistency across months, and runs lightweight sanity checks (including case-collision safety for `user_id → user_key`).

**This notebook does not write outputs**; it only reads and prints.


In [ ]:
# --- Bootstrap (release-aware) ---
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))  # make repo root importable when running from notebooks/
from notebooks.nb_paths import bootstrap

# Optionally set a release override here (e.g., "r5.1").
RELEASE_OVERRIDE = None  # e.g., "r5.1"

env = bootstrap(RELEASE_OVERRIDE)
LDAP_DIR = env.RAW / "LDAP"

print(f"RELEASE = {env.RELEASE}")
print(f"LDAP_DIR = {LDAP_DIR}")
if not LDAP_DIR.exists():
    raise FileNotFoundError(f"LDAP folder not found: {LDAP_DIR}")

paths = sorted(LDAP_DIR.glob("*.csv"))
print(f"Found {len(paths)} LDAP CSV(s). Example:")
for p in paths[:5]:
    print("  -", p.name)
if not paths:
    raise FileNotFoundError("No LDAP CSV files present.")


## Header audit per month

In [ ]:

from collections import defaultdict

def read_headers(p: Path):
    try:
        df = pd.read_csv(p, nrows=0, dtype=str)
        return list(map(str, df.columns))
    except Exception as e:
        return [f"<error: {e}>"]

headers_by_file = {p.name: read_headers(p) for p in paths}

print("Per-file headers (first 100 chars for each col list):")
for name, cols in headers_by_file.items():
    print(f"- {name}: {cols}")

# Build sets for comparison
sets = {name: tuple(cols) for name, cols in headers_by_file.items()}
unique_header_signatures = {}
for name, cols in sets.items():
    unique_header_signatures.setdefault(cols, []).append(name)

print("\nUnique header signatures:")
for i, (cols, files) in enumerate(unique_header_signatures.items(), 1):
    print(f"[sig {i}] cols={list(cols)}")
    print("  files:", ", ".join(files))


## Required column check

In [ ]:

REQUIRED = {"user_id"}  # minimally required for downstream joins
OPTIONAL = {
    "employee_name","email","role",
    "business_unit","functional_unit","department","team","supervisor",
}

missing_map = {}
for name, cols in headers_by_file.items():
    missing = sorted(list(REQUIRED - set(c.lower() for c in cols)))
    if missing:
        missing_map[name] = missing

if missing_map:
    print("**ERROR**: Required columns missing in some files:")
    for f, miss in missing_map.items():
        print(" -", f, "missing:", miss)
    raise AssertionError("Required columns missing; fix raw inputs or header mapping before ETL.")
else:
    print("OK: All files contain required columns:", sorted(REQUIRED))


## Case-collision safety for `user_id → user_key`

In [ ]:

from src.helpers.users import normalize_user_series

n_rows = 0
orig_uniques = set()
low_uniques = set()

use = ["user_id"]
for p in paths:
    try:
        s = pd.read_csv(p, dtype=str, usecols=use)["user_id"].dropna().astype(str)
        n_rows += len(s)
        orig_uniques.update(s.unique().tolist())
        low_uniques.update(normalize_user_series(s).dropna().unique().tolist())
    except Exception as e:
        print(f"warn: could not read user_id from {p.name}: {e}")

print("Rows scanned:", n_rows)
print("Unique user_id (orig):", len(orig_uniques))
print("Unique user_id (low) :", len(low_uniques))
collisions = max(0, len(orig_uniques) - len(low_uniques))
print("Case-only collisions :", collisions)
assert collisions == 0, "Case-only collisions found; do NOT lowercase user_id."
print("OK: Lowercasing to user_key is collision-free in this release.")


## Null-rate profile (sampled by month)

In [ ]:

# We sample a couple of files to avoid loading everything.
sample_files = paths[:3] if len(paths) > 3 else paths
frames = []
for p in sample_files:
    try:
        df = pd.read_csv(p, dtype=str)
        df["__file__"] = p.name
        frames.append(df)
    except Exception as e:
        print("warn:", p.name, "->", e)

if frames:
    samp = pd.concat(frames, ignore_index=True)
    # Uniform lowercasing for column names
    samp.columns = [c.strip() for c in samp.columns]
    null_rates = samp.isna().mean().sort_values(ascending=False)
    print("Null-rate (sampled files):")
    print((null_rates * 100).round(2).to_frame("null_%").head(50))
else:
    print("No sample frames loaded; skipping null-rate profile.")


## Value samples (role & org fields)

In [ ]:

CATS = ["role","business_unit","functional_unit","department","team","supervisor"]

def value_preview(df: pd.DataFrame, col: str, top: int = 10):
    if col not in df.columns:
        print(f"- {col}: (not present)")
        return
    vc = df[col].value_counts(dropna=True).head(top)
    print(f"- {col}: top {top}")
    if vc.empty:
        print("  (no non-null values)")
    else:
        for k, v in vc.items():
            print(f"  {k!r}: {v}")

if frames:
    for col in CATS:
        value_preview(samp, col, top=10)
else:
    print("No sample frames; skipping value previews.")


## Column normalization suggestion (print-only)

In [ ]:

# Based on headers present, suggest the keep-set for snapshots/as-of.
present_cols = set()
for cols in headers_by_file.values():
    present_cols.update(map(str, cols))

keep_snapshots = ["user_id", "employee_name", "email", "role",
                  "business_unit","functional_unit","department","team","supervisor"]
keep_snapshots = [c for c in keep_snapshots if c in present_cols]

print("Suggested keep columns for snapshots (pre-ETL):")
print(keep_snapshots)

print("\nNote: ETL will create derived columns like 'user_key' (lowercased user_id) and 'snapshot_date'.")
print("As-of (per month) will reduce to ['user_key','event_month', ...present org fields...]")


In [ ]:
# %% [markdown]
# ## Supervisor sanity: flag any email-like values (contains '@')

# %%
import pandas as pd
from pathlib import Path
import re

# Reuse existing env/LDAP_DIR if present; otherwise bootstrap quickly.
try:
    LDAP_DIR
except NameError:
    from notebooks.nb_paths import bootstrap
    env = bootstrap()
    LDAP_DIR = env.RAW / "LDAP"

paths = sorted(Path(LDAP_DIR).glob("*.csv"))
if not paths:
    raise FileNotFoundError(f"No LDAP CSVs under {LDAP_DIR}")

email_like_pat = re.compile(r"@")  # simple, exactly what you asked for
rows_scanned = 0
rows_with_supervisor = 0
violations_total = 0

report = []  # (file, total_rows, with_supervisor, email_like_count, examples)

for p in paths:
    try:
        # Only pull the supervisor column; stay light on memory.
        df = pd.read_csv(p, usecols=["supervisor"], dtype=str, on_bad_lines="warn", encoding="utf-8")
    except ValueError:
        # file has no 'supervisor' column
        report.append((p.name, 0, 0, 0, []))
        continue

    s = df["supervisor"].dropna().astype(str).str.strip()
    rows_scanned        += len(df)
    rows_with_supervisor += len(s)

    mask = s.str.contains(email_like_pat)
    n_bad = int(mask.sum())
    violations_total += n_bad

    examples = s[mask].head(10).tolist()
    report.append((p.name, len(df), len(s), n_bad, examples))

# Pretty print
print("=== LDAP Supervisor Email-Like Scan (simple '@' test) ===")
for name, n_total, n_sup, n_bad, examples in report:
    if n_total == 0 and n_sup == 0:
        print(f"- {name}: (no 'supervisor' column)")
        continue
    rate = (n_bad / n_sup * 100.0) if n_sup else 0.0
    print(f"- {name}: rows={n_total:,} | with_supervisor={n_sup:,} | email-like={n_bad:,} ({rate:.3f}%)")
    if n_bad and examples:
        print("    examples:", "; ".join(map(str, examples)))

print("\nTotals:")
print(f"  files scanned           : {len(paths)}")
print(f"  rows scanned            : {rows_scanned:,}")
print(f"  rows with supervisor    : {rows_with_supervisor:,}")
print(f"  email-like supervisors  : {violations_total:,} "
      f"({(violations_total / rows_with_supervisor * 100.0 if rows_with_supervisor else 0.0):.3f}%)")

# Optional: hard-fail right here if any email-like values are present
assert violations_total == 0, "Found '@' in supervisor values. See per-file report above."


---

**Next step:** If everything looks good -> run CLI:

```bash
python -m scripts.etl ldap --profile full --family ldap_v3_full
python -m scripts.etl ldap --profile lean --family ldap_v3_lean
```


In [ ]:
# --- Inspect LDAP outputs (lean + full) using env paths ---
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

print("RELEASE:", env.RELEASE)
print("PROJECT:", env.PROJECT)
print("OUT dir:", env.OUT)

families = ("ldap_v3_full", "ldap_v3_lean")
files = ("ldap_snapshots.parquet", "ldap_asof_by_month.parquet")

def _cols(p): 
    return pq.ParquetFile(p).schema.names

def _nrows(p):
    return pq.ParquetFile(p).metadata.num_rows

def _safe_head(p, n=5):
    try:
        return pd.read_parquet(p, nrows=n)
    except TypeError:
        df = pd.read_parquet(p)
        return df.head(n)

for fam in families:
    base = env.OUT / env.RELEASE / fam   # <<< key change
    print(f"\n=== {fam} ===")
    print("Base:", base)
    for name in files:
        p = base / name
        if not p.exists():
            print(f"[MISSING] {p}")
            continue

        cols = _cols(p)
        print(f"\n{name}")
        print("• columns:", cols)
        print("• rows   :", _nrows(p))

        time_col = "snapshot_date" if "snapshots" in name else "event_month"
        if time_col in cols:
            t = pd.read_parquet(p, columns=[time_col])[time_col]
            t = pd.to_datetime(t, errors="coerce")
            print(f"• {time_col} range:", t.min(), "→", t.max())

        print("• head:")
        display(_safe_head(p, 5))

        key_cols = ["user_key", "snapshot_date"] if "snapshots" in name else ["user_key","event_month"]
        if all(c in cols for c in key_cols):
            kdf = pd.read_parquet(p, columns=key_cols)
            dup = int(kdf.duplicated(key_cols).sum())
            print(f"• duplicates on {tuple(key_cols)}:", dup)

# Quick “what exists” dump if still missing
print("\nExisting under OUT/RELEASE:")
for p in sorted((env.OUT/env.RELEASE).glob("**/*.parquet"))[:20]:
    print(" -", p.relative_to(env.PROJECT))

In [ ]:
# --- LDAP email domain audit (lightweight, CSV-only, read-only) ---
from pathlib import Path
import pandas as pd
from collections import Counter
import re

# Reuse notebook vars if present; otherwise resolve quickly
try:
    LDAP_DIR
except NameError:
    try:
        from notebooks.nb_paths import bootstrap
    except Exception:
        from nb_paths import bootstrap  # fallback if PYTHONPATH set
    env = bootstrap()
    LDAP_DIR = env.RAW / "LDAP"

paths = sorted(Path(LDAP_DIR).glob("*.csv"))
if not paths:
    raise FileNotFoundError(f"No LDAP CSVs under {LDAP_DIR}")

total_rows = 0
rows_with_email = 0
internal_rows = 0
non_internal_domains = Counter()
email_pat = re.compile(r"@([\w\.-]+)$")

# Pass 1: only read 'email' when present; skip files that lack it
for p in paths:
    # quick header peek
    header = pd.read_csv(p, nrows=0, dtype=str).columns.str.lower().tolist()
    if "email" not in header:
        continue

    s = pd.read_csv(p, usecols=["email"], dtype=str, on_bad_lines="warn")["email"]
    total_rows += len(s)

    s = s.dropna().astype(str).str.strip().str.lower()
    rows_with_email += int(s.notna().sum())

    # extract domain (keeps only rows that actually have an '@something')
    dom = s.str.extract(email_pat, expand=False)
    # count internal
    internal_mask = dom.eq("dtaa.com")
    internal_rows += int(internal_mask.fillna(False).sum())

    # count non-internal domains
    non_int = dom[~internal_mask & dom.notna()]
    non_internal_domains.update(non_int.tolist())

# Report
print("=== LDAP Email Domain Audit (CSV) ===")
print(f"files scanned         : {len(paths)}")
print(f"rows total (email-col files only): {total_rows:,}")
print(f"rows with email       : {rows_with_email:,}")
if rows_with_email:
    internal_pct = round(100.0 * internal_rows / rows_with_email, 2)
else:
    internal_pct = 0.0
print(f"internal (@dtaa.com)  : {internal_rows:,} ({internal_pct}%)")

if non_internal_domains:
    print("\n[WARN] Non-internal domains detected (top 15):")
    for dom, n in non_internal_domains.most_common(15):
        print(f"  {dom:25} {n:,}")
else:
    print("\n[OK] All observed LDAP emails are internal (@dtaa.com).")

In [ ]:
# --- Validate `is_admin` against LDAP roles using project-root paths ---
import pandas as pd
from pathlib import Path

try:
    from notebooks.nb_paths import bootstrap
except ImportError:
    from nb_paths import bootstrap  # fallback if you've added PYTHONPATH=.

# Honor your optional override; otherwise read release.txt
env = bootstrap(release=(RELEASE_OVERRIDE or None))
base = Path(env.OUT) / env.RELEASE  # e.g., <repo>/out/r5.1

# Probe families under the *real* out/<release>/ directory
snap_candidates = [
    base / "ldap_v3_full" / "ldap_snapshots.parquet",
    base / "ldap_v3"      / "ldap_snapshots.parquet",
    base / "ldap_v3_lean" / "ldap_snapshots.parquet",
]
asof_candidates = [
    base / "ldap_v3_lean" / "ldap_asof_by_month.parquet",
    base / "ldap_v3"      / "ldap_asof_by_month.parquet",
    base / "ldap_v3_full" / "ldap_asof_by_month.parquet",
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

snap_path = first_existing(snap_candidates)
asof_path = first_existing(asof_candidates)

if not snap_path or not asof_path:
    print("Could not find required parquet(s). Looked for:")
    print("  snapshots:", *map(str, snap_candidates), sep="\n    - ")
    print("  as-of    :", *map(str, asof_candidates), sep="\n    - ")
    # Helpful: show what *does* exist under out/<release>
    print("\nExisting under", base, ":")
    for p in sorted(base.rglob("*.parquet")):
        print("   -", p.relative_to(base))
else:
    print("[using]")
    print("  snapshots ->", snap_path)
    print("  as-of     ->", asof_path)

    snap = pd.read_parquet(snap_path)
    asof = pd.read_parquet(asof_path)

    # Guardrails
    need_snap = {"user_key","snapshot_date","role"}
    need_asof = {"user_key","event_month","is_admin"}
    missing_snap = need_snap - set(map(str, snap.columns))
    missing_asof = need_asof - set(map(str, asof.columns))
    if missing_snap or missing_asof:
        raise AssertionError(f"Missing columns — snapshots: {sorted(missing_snap)}; as-of: {sorted(missing_asof)}")

    # Normalize for month alignment
    snap = snap.copy()
    snap["event_month"] = pd.to_datetime(snap["snapshot_date"]).dt.to_period("M").dt.to_timestamp()
    snap["role_l"] = snap["role"].astype(str).str.strip().str.lower()

    it_rows = snap.loc[snap["role_l"] == "itadmin", ["user_key","event_month"]].drop_duplicates()
    flag_rows = asof.loc[asof["is_admin"] == True, ["user_key","event_month"]].drop_duplicates()

    # Differences in both directions
    missing = (
        it_rows.merge(flag_rows, on=["user_key","event_month"], how="left", indicator=True)
               .query("_merge == 'left_only'").drop(columns="_merge")
    )
    extra = (
        flag_rows.merge(it_rows, on=["user_key","event_month"], how="left", indicator=True)
                 .query("_merge == 'left_only'").drop(columns="_merge")
    )

    print("\nCounts")
    print("------")
    print("ITAdmin rows in snapshots:", len(it_rows))
    print("is_admin=True in as-of   :", len(flag_rows))
    print("Missing (ITAdmin but not flagged):", len(missing))
    print("Extra (flagged but role≠ITAdmin):", len(extra))

    if len(missing) or len(extra):
        print("\nExamples (up to 10 each)")
        if len(missing): display(missing.head(10))
        if len(extra):   display(extra.head(10))
    else:
        print("\n✅ is_admin matches ITAdmin exactly.")

In [ ]:
# LDAP QC: load -> sanity checks -> email/domain audit -> role/is_admin consistency -> joinability -> null audit
from pathlib import Path
import pandas as pd

# --- Resolve release + candidate artifact paths ---
try:
    from notebooks.nb_paths import bootstrap
except ImportError:
    from nb_paths import bootstrap

env = bootstrap(release=(RELEASE_OVERRIDE or None))
base = Path(env.OUT) / env.RELEASE

CAND_SNAP = [
    base / "ldap_v3_full" / "ldap_snapshots.parquet",
    base / "ldap_v3"      / "ldap_snapshots.parquet",
    base / "ldap_v3_lean" / "ldap_snapshots.parquet",
]
CAND_ASOF = [
    base / "ldap_v3_lean" / "ldap_asof_by_month.parquet",
    base / "ldap_v3"      / "ldap_asof_by_month.parquet",
    base / "ldap_v3_full" / "ldap_asof_by_month.parquet",
]
def _pick(existing): 
    for p in existing:
        if p.exists(): 
            return p
    return None

snap_p = _pick(CAND_SNAP)
asof_p = _pick(CAND_ASOF)
if not snap_p or not asof_p:
    raise FileNotFoundError(
        "Could not find required parquet(s).\n"
        f"  snapshots: {[str(p) for p in CAND_SNAP]}\n"
        f"  as-of    : {[str(p) for p in CAND_ASOF]}"
    )

print(f"Loaded:\n  snapshots -> {snap_p}\n  as-of     -> {asof_p}")
snap = pd.read_parquet(snap_p)
asof = pd.read_parquet(asof_p)

# --- Columns present (minimal) ---
need_snap = {"user_key","snapshot_date","email","role"}
need_asof = {"user_key","event_month","email","role","is_admin"}
miss_snap = need_snap - set(map(str, snap.columns))
miss_asof = need_asof - set(map(str, asof.columns))
assert not miss_snap, f"missing in snapshots: {sorted(miss_snap)}"
assert not miss_asof, f"missing in asof: {sorted(miss_asof)}"
print("✓ required columns present.")

# --- Key uniqueness in snapshots ---
dup_pairs = snap.duplicated(["user_key","snapshot_date"]).sum()
assert dup_pairs == 0, f"duplicate (user_key,snapshot_date) rows: {dup_pairs}"
print("✓ unique (user_key, snapshot_date).")

# --- Email normalization + external domains (quick peek) ---
non_lower = snap["email"].dropna().loc[lambda s: s.str.contains(r"[A-Z]")].head(1)
assert non_lower.empty, "email contains uppercase; normalization failed."
assert not (snap["email"] == "").any(), "empty-string emails; expected NA instead."
domains = snap["email"].dropna().str.extract(r"@([\w\.-]+)$")[0]
ext = domains[domains.ne("dtaa.com")].value_counts().head(10)
print("External domains in LDAP (top 10):")
print(ext if not ext.empty else "  (none)")
print("✓ email normalization OK.")

# --- Role sanity (source for is_admin) ---
role_l = snap["role"].astype(str).str.strip().str.lower()
assert role_l.notna().any(), "no role values?"
print("Role sample:\n", role_l.value_counts().head(10).to_string())

# --- is_admin consistency with role == ITAdmin (month-aligned) ---
ss = snap.copy()
ss["event_month"] = pd.to_datetime(ss["snapshot_date"]).dt.to_period("M").dt.to_timestamp(how="start")
ss["role_l"] = ss["role"].astype(str).str.strip().str.lower()
it_rows = ss.loc[ss["role_l"].eq("itadmin"), ["user_key","event_month"]].drop_duplicates()
flag_rows = asof.loc[asof["is_admin"]==True, ["user_key","event_month"]].drop_duplicates()

missing = (it_rows.merge(flag_rows, on=["user_key","event_month"], how="left", indicator=True)
                 .query("_merge=='left_only'")[["user_key","event_month"]])
extra   = (flag_rows.merge(it_rows, on=["user_key","event_month"], how="left", indicator=True)
                 .query("_merge=='left_only'")[["user_key","event_month"]])

print(f"ITAdmin pairs (snapshots): {len(it_rows):,}")
print(f"is_admin=True (as-of)    : {len(flag_rows):,}")
print(f"Missing (ITAdmin not flagged): {len(missing):,}")
print(f"Extra (flagged but role≠ITAdmin): {len(extra):,}")
assert len(missing)==0 and len(extra)==0, "is_admin mismatch; inspect `missing` and `extra`."
print("✓ is_admin matches ITAdmin exactly.")

# --- Joinability spot-check (downstream sanity) ---
sample_keys = asof[["user_key","event_month"]].dropna().head(3)
assert len(sample_keys) > 0, "no join keys present in as-of?"
print("Sample as-of join keys:\n", sample_keys.to_string(index=False))
print("✓ as-of join keys look good.")

# --- Null audit (critical fields) ---
crit = ["user_key","snapshot_date","email","role"]
null_rates = (snap[crit].isna().mean().mul(100).round(2)).astype(str) + "%"
print("Null rates (snapshots):\n", null_rates.to_string())

print("\nLDAP QC ✓ All checks passed.")

In [ ]:
from notebooks.nb_paths import bootstrap
from src.helpers.io import out_path
import pandas as pd
env = bootstrap()
p_full = out_path(env, "ldap_v3_full", "ldap_asof_by_month")
p_lean = out_path(env, "ldap_v3_lean", "ldap_asof_by_month")
display(pd.read_parquet(p_full, columns=["user_key","event_month","supervisor","supervisor_key","last_seen"]).head(10))
display(pd.read_parquet(p_lean, columns=["user_key","event_month","email","role","is_admin","team","supervisor_key","last_seen"]).head(10))